<a href="https://colab.research.google.com/github/Datadog-995/Cleaned-Butcher-Sales-Portfolio/blob/main/7_financial_transactions_final_cleanedi_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
df_reloaded = pd.read_csv('7-financial_Transactions_Final_Clean.csv')
print(df_reloaded.head())

  Transaction_ID Transaction_Date Customer_ID    Product_Name  Quantity  \
0          T0001       2024-08-02       C2205      Headphones       5.0   
1          T0002       2020-02-10       C3156         Coffee      469.0   
2          T0003       1900-01-01       C2919          Tablet       4.0   
3          T0004       2020-08-17       C3009             Tab       7.0   
4          T0005       1900-01-01       C3488  Coffee Machine      10.0   

        Price Payment_Method Transaction_Status  
0    0.000000        pay pal            Unknown  
1 -445.342025     creditcard            Pending  
2  810.993012    credit card          Completed  
3  868.608341         PayPal            Pending  
4 -763.122449         PayPal          Completed  


In [1]:
import pandas as pd

# 1. Load the dirty dataset
# Replace with your actual local file path if different
# The original file 'dirty_financial_transactions.csv' was not found.
# Using 'dirty_financial_transactions 2.csv' which is available in the environment.
df = pd.read_csv("7-dirty_financial_transactions.csv")

# 2. Basic Shape & Data Type Audit
print("--- DATASET SHAPE ---")
print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}\n")

print("--- DATA TYPE INITIAL AUDIT ---")
print(df.dtypes)
print("\n--- MISSING VALUE MAP ---")
print(df.isnull().sum())

--- DATASET SHAPE ---
Total Rows: 100000, Total Columns: 8

--- DATA TYPE INITIAL AUDIT ---
Transaction_ID         object
Transaction_Date       object
Customer_ID            object
Product_Name           object
Quantity              float64
Price                  object
Payment_Method         object
Transaction_Status     object
dtype: object

--- MISSING VALUE MAP ---
Transaction_ID         5018
Transaction_Date       4880
Customer_ID            4878
Product_Name              0
Quantity               5019
Price                 33497
Payment_Method            0
Transaction_Status    16679
dtype: int64


## 3. Data Cleaning and Preprocessing

In [2]:
# Convert 'Transaction_Date' to datetime
# Using errors='coerce' will turn unparseable dates into NaT (Not a Time)
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'], errors='coerce')

# Convert 'Price' to numeric, handling errors
# 'errors='coerce' will turn non-numeric values into NaN
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# Convert 'Quantity' to numeric (it's already float, but good for consistency)
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')

print("--- DATA TYPES AFTER CONVERSION ---")
print(df.dtypes)

print("\n--- MISSING VALUES AFTER CONVERSION ---")
print(df.isnull().sum())

--- DATA TYPES AFTER CONVERSION ---
Transaction_ID                object
Transaction_Date      datetime64[ns]
Customer_ID                   object
Product_Name                  object
Quantity                     float64
Price                        float64
Payment_Method                object
Transaction_Status            object
dtype: object

--- MISSING VALUES AFTER CONVERSION ---
Transaction_ID         5018
Transaction_Date      68261
Customer_ID            4878
Product_Name              0
Quantity               5019
Price                 40137
Payment_Method            0
Transaction_Status    16679
dtype: int64


### Handle Missing Values

In [3]:
# The user wants to keep all data and not remove any missing values.
# Instead of dropping, we will fill missing values with appropriate placeholders.

# Fill missing 'Transaction_ID' and 'Customer_ID' with 'Unknown'
# Addressing FutureWarning by direct assignment instead of inplace=True
df['Transaction_ID'] = df['Transaction_ID'].fillna('Unknown')
df['Customer_ID'] = df['Customer_ID'].fillna('Unknown')

# Fill missing 'Transaction_Date' (NaT values) with a placeholder date (e.g., 1900-01-01)
# This keeps the column as datetime, allowing for date-based operations later if needed.
# Addressing FutureWarning by direct assignment instead of inplace=True
df['Transaction_Date'] = df['Transaction_Date'].fillna(pd.Timestamp('1900-01-01'))

# Fill missing 'Quantity' and 'Price' with 0, assuming a missing value means no quantity/price
# Addressing FutureWarning by direct assignment instead of inplace=True
df['Quantity'] = df['Quantity'].fillna(0)
df['Price'] = df['Price'].fillna(0)

print("\n--- MISSING VALUES AFTER FILLING (NO DROPS) ---")
print(df.isnull().sum())


--- MISSING VALUES AFTER FILLING (NO DROPS) ---
Transaction_ID            0
Transaction_Date          0
Customer_ID               0
Product_Name              0
Quantity                  0
Price                     0
Payment_Method            0
Transaction_Status    16679
dtype: int64


### Clean and Standardize `Transaction_Status`

In [4]:
# Standardize 'Transaction_Status' to a consistent format
df['Transaction_Status'] = df['Transaction_Status'].str.strip().str.capitalize()

# Fill missing 'Transaction_Status' with 'Unknown'
# Addressing FutureWarning by direct assignment instead of inplace=True
df['Transaction_Status'] = df['Transaction_Status'].fillna('Unknown')

print("\n--- UNIQUE TRANSACTION STATUS VALUES ---")
print(df['Transaction_Status'].unique())

print("\n--- MISSING VALUES AFTER STATUS CLEANING ---")
print(df.isnull().sum())


--- UNIQUE TRANSACTION STATUS VALUES ---
['Unknown' 'Pending' 'Completed' 'Complete' 'Failed']

--- MISSING VALUES AFTER STATUS CLEANING ---
Transaction_ID        0
Transaction_Date      0
Customer_ID           0
Product_Name          0
Quantity              0
Price                 0
Payment_Method        0
Transaction_Status    0
dtype: int64


### Address Inconsistencies: Negative Quantities

In [5]:
# Identify and handle negative quantities
negative_quantities = df[df['Quantity'] < 0]
print(f"Number of transactions with negative quantities: {len(negative_quantities)}")

# Option 1: Convert negative quantities to their absolute values (assuming they represent returns or cancellations with positive magnitude)
# df['Quantity'] = df['Quantity'].abs()

# Option 2: Drop rows with negative quantities if they are considered invalid data entries.
# Given the nature of transactions, a negative quantity might imply a return or correction.
# For simplicity, let's convert them to positive values, but this depends on business logic.
df['Quantity'] = df['Quantity'].abs()

print("\n--- DATASET SHAPE AFTER CLEANING ---")
print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}")

print("\n--- FINAL DATA TYPES ---")
print(df.dtypes)

print("\n--- FINAL MISSING VALUE MAP ---")
print(df.isnull().sum())

Number of transactions with negative quantities: 31619

--- DATASET SHAPE AFTER CLEANING ---
Total Rows: 100000, Total Columns: 8

--- FINAL DATA TYPES ---
Transaction_ID                object
Transaction_Date      datetime64[ns]
Customer_ID                   object
Product_Name                  object
Quantity                     float64
Price                        float64
Payment_Method                object
Transaction_Status            object
dtype: object

--- FINAL MISSING VALUE MAP ---
Transaction_ID        0
Transaction_Date      0
Customer_ID           0
Product_Name          0
Quantity              0
Price                 0
Payment_Method        0
Transaction_Status    0
dtype: int64


### Display cleaned data sample

In [6]:
display(df.head())

,Transaction_ID,Transaction_Date,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status
0,T0001,2024-08-02,C2205,Headphones,5.0,0.000000,pay pal,Unknown
1,T0002,2020-02-10,C3156,Coffee,469.0,-445.342025,creditcard,Pending
2,T0003,1900-01-01,C2919,Tablet,4.0,810.993012,credit card,Completed
3,T0004,2020-08-17,C3009,Tab,7.0,868.608341,PayPal,Pending
4,T0005,1900-01-01,C3488,Coffee Machine,10.0,-763.122449,PayPal,Completed


In [7]:
# 1. Save the ENTIRE dataframe to a CSV without the index column
df.to_csv("7-financial_Transactions_Final_Clean.csv", index=False)

# 2. Automatically copy this full file to your Google Drive for safe keeping
import shutil
shutil.copy("7-financial_Transactions_Final_Clean.csv", "/content/drive/My Drive/7-financial_Transactions_Final_Clean.csv")

print("Success! The full dataset is now saved and ready to download.")

Success! The full dataset is now saved and ready to download.


In [8]:
# 1. Save the ENTIRE dataframe to a CSV without the index column
df.to_csv("7-financial_Transactions_Final_Clean.csv", index=False)

# 2. Automatically copy this full file to your Google Drive for safe keeping
import shutil
shutil.copy("7-financial_Transactions_Final_Clean.csv", "/content/drive/My Drive/7-financial_Transactions_Final_Clean.csv")

print("Success! The full dataset is now saved and ready to download.")

Success! The full dataset is now saved and ready to download.


In [9]:
# 1. Standardize the remaining headers to use underscores
df.rename(columns={
    'Quantity Audit Check': 'Quantity_Audit_Check',
    'Business Rule Check': 'Business_Rule_Check'
}, inplace=True)

# 2. Drop the redundant bad column (the one with spaces) if it exists
if 'Price Audit Check' in df.columns:
    df.drop(columns=['Price Audit Check'], inplace=True)

# 3. Verify the final columns look perfectly aligned
print(df.columns)

Index(['Transaction_ID', 'Transaction_Date', 'Customer_ID', 'Product_Name',
       'Quantity', 'Price', 'Payment_Method', 'Transaction_Status'],
      dtype='object')


In [10]:
# Export the entire dataset without any row limits
df.to_csv("7-financial_Transactions_FULL_100k.csv", index=False)

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
# 1. Clean the full DataFrame across all 100,000 rows
df['Price'] = df['Price'].abs()

# Standardize Payment Methods (stripping spaces and lowercase)
df['Payment_Method'] = df['Payment_Method'].str.replace(' ', '').str.lower()

# Map truncated product names globally
product_mapping = {
    'Cof': 'Coffee',
    'Coffee Ma': 'Coffee Machine',
    'Coffee M': 'Coffee Machine',
    'Smar': 'Smartphone',
    'Smartp': 'Smartphone',
    'Tab': 'Tablet',
    'T': 'Tablet',
    'Headp': 'Headphones'
}
df['Product_Name'] = df['Product_Name'].replace(product_mapping)

# Standardize remaining transaction statuses
status_mapping = {'Complete': 'Completed'}
df['Transaction_Status'] = df['Transaction_Status'].replace(status_mapping)

# 2. Verify the full size remains intact
print(f"Verification - Total cleaned rows: {len(df)}")

# 3. Export the UNLIMITED, fully cleaned dataset directly to your Drive
output_path = '/content/drive/My Drive/7-financial_Transactions_Final_Clean.csv'
df.to_csv(output_path, index=False)

print(f"Success! The entire dataset has been fully cleaned and saved to: {output_path}")

Verification - Total cleaned rows: 100000
Success! The entire dataset has been fully cleaned and saved to: /content/drive/My Drive/7-financial_Transactions_Final_Clean.csv


In [12]:
df.to_csv("7-financial_Transactions_FULL_100k.csv", index=False)

In [21]:
# 1. Ensure we are working with the full 100k dataset
# Load the original full file to guarantee we aren't using a truncated version
df_full = pd.read_csv("7-dirty_financial_transactions.csv")

# 2. Apply all cleaning steps across the entire UNLIMITED dataset
df_full['Transaction_ID'] = df_full['Transaction_ID'].fillna('Unknown')
df_full['Customer_ID'] = df_full['Customer_ID'].fillna('Unknown')
df_full['Transaction_Date'] = pd.to_datetime(df_full['Transaction_Date'], errors='coerce').fillna(pd.Timestamp('1900-01-01'))

# Fix missing values and absolute values for numeric columns
df_full['Quantity'] = pd.to_numeric(df_full['Quantity'], errors='coerce').fillna(0).abs()
df_full['Price'] = pd.to_numeric(df_full['Price'], errors='coerce').fillna(0).abs()

# Standardize Payment Methods (lowercase and strip spaces)
df_full['Payment_Method'] = df_full['Payment_Method'].astype(str).str.replace(' ', '').str.lower()

# Map truncated product names globally across all rows
product_mapping = {
    'Cof': 'Coffee',
    'Coffee Ma': 'Coffee Machine',
    'Coffee M': 'Coffee Machine',
    'Smar': 'Smartphone',
    'Smartp': 'Smartphone',
    'Tab': 'Tablet',
    'T': 'Tablet',
    'Headp': 'Headphones'
}
df_full['Product_Name'] = df_full['Product_Name'].replace(product_mapping)

# Standardize transaction statuses
df_full['Transaction_Status'] = df_full['Transaction_Status'].astype(str).str.strip().str.capitalize()
status_mapping = {'Complete': 'Completed'}
df_full['Transaction_Status'] = df_full['Transaction_Status'].replace(status_mapping)

# 3. CRITICAL VERIFICATION: Print the exact row count of the final dataset
print("==================================================")
print(f"VERIFICATION - Total Cleaned Rows: {df_full.shape[0]}")
print("==================================================")

# 4. Save the full, unlimited file back to your environment and Google Drive
df_full.to_csv("7-financial_Transactions_Final_Clean.csv", index=False)

import shutil
shutil.copy("7-financial_Transactions_Final_Clean.csv", "/content/drive/My Drive/7-financial_Transactions_Final_Clean.csv")

print("Success! The entire 100k dataset is fully cleaned and saved to Google Drive.")

VERIFICATION - Total Cleaned Rows: 100000
Success! The entire 100k dataset is fully cleaned and saved to Google Drive.


In [22]:
import pandas as pd
df_full = pd.read_csv("7-financial_Transactions_FULL_100k.csv")
df_full['Price'] = pd.to_numeric(df_full['Price'], errors='coerce').fillna(0).abs()
df_full['Payment_Method'] = df_full['Payment_Method'].astype(str).str.lower().str.strip()
product_mapping = {
    'Smar': 'Smartphone', 'Smartp': 'Smartphone', 'Smartpho': 'Smartphone',
    'Hea': 'Headphones', 'Headp': 'Headphones', 'Headpho': 'Headphones',
    'Lap': 'Laptop', 'Lapt': 'Laptop', 'Lapto': 'Laptop',
    'Tab': 'Tablet', 'Tabl': 'Tablet', 'Table': 'Tablet',
    'T': 'Tablet',
    'Cof': 'Coffee Machine', 'Coffe': 'Coffee Machine', 'Coffee M': 'Coffee Machine', 'Coffee Ma': 'Coffee Machine'
}
df_full['Product_Name'] = df_full['Product_Name'].replace(product_mapping)
status_mapping = {'Complete': 'Completed'}
df_full['Transaction_Status'] = df_full['Transaction_Status'].astype(str).str.strip().replace(status_mapping)
df_full.to_csv("/content/drive/My Drive/7-financial_Transactions_Final_Clean.csv", index=False)
print("Verification - Total Cleaned Rows:", df_full.shape[0])


Verification - Total Cleaned Rows: 100000


In [24]:
# 1. Create the audit flags based on your data cleaning rules
# If the date is 1900-01-01 (your fallback for missing/unparseable), mark it appropriately
df_full['audited_dates'] = df_full['Transaction_Date'].apply(
    lambda x: 'Missing/Invalid' if x == pd.Timestamp('1900-01-01') else 'Valid'
)

# 2. Move the column so it sits directly next to 'Transaction_Date'
columns = list(df_full.columns)
# Remove it from the end of the list
columns.remove('audited_dates')
# Find the position of Transaction_Date and insert it right after (index + 1)
date_idx = columns.index('Transaction_Date')
columns.insert(date_idx + 1, 'audited_dates')

# Reorder the DataFrame columns
df_full = df_full[columns]

# 3. Save the updated layout back to your Drive
df_full.to_csv("7-financial_Transactions_Final_Clean.csv", index=False)
df_full.to_csv("/content/drive/My Drive/7-financial_Transactions_Final_Clean.csv", index=False)

# Preview the new column placement
print("Columns layout:", df_full.columns.tolist())
df_full[['Transaction_ID', 'Transaction_Date', 'audited_dates', 'Customer_ID']].head()

Columns layout: ['Transaction_ID', 'Transaction_Date', 'audited_dates', 'Customer_ID', 'Product_Name', 'Quantity', 'Price', 'Payment_Method', 'Transaction_Status']


,Transaction_ID,Transaction_Date,audited_dates,Customer_ID
0,T0001,2024-08-02,Valid,C2205
1,T0002,2020-02-10,Valid,C3156
2,T0003,1900-01-01,Valid,C2919
3,T0004,2020-08-17,Valid,C3009
4,T0005,1900-01-01,Valid,C3488


In [25]:
df_full.head()

,Transaction_ID,Transaction_Date,audited_dates,Customer_ID,Product_Name,Quantity,Price,Payment_Method,Transaction_Status
0,T0001,2024-08-02,Valid,C2205,Headphones,5.0,0.000000,pay pal,Unknown
1,T0002,2020-02-10,Valid,C3156,Coffee,469.0,445.342025,creditcard,Pending
2,T0003,1900-01-01,Valid,C2919,Tablet,4.0,810.993012,credit card,Completed
3,T0004,2020-08-17,Valid,C3009,Tablet,7.0,868.608341,paypal,Pending
4,T0005,1900-01-01,Valid,C3488,Coffee Machine,10.0,763.122449,paypal,Completed


In [26]:
# 1. Reload original file data to catch raw missing/negative prices
df_full = pd.read_csv("7-dirty_financial_transactions.csv")

# 2. Audit the raw Price column BEFORE cleaning it
# Coerce to numeric first just to identify unparseable text vs actual numbers
raw_price_numeric = pd.to_numeric(df_full['Price'], errors='coerce')

def audit_price(val, numeric_val):
    if pd.isna(val) or pd.isna(numeric_val):
        return 'Missing/Invalid'
    elif numeric_val < 0:
        return 'Negative'
    else:
        return 'Valid'

df_full['audited_prices'] = df_full.apply(lambda row: audit_price(row['Price'], raw_price_numeric[row.name]), axis=1)

# 3. Clean the standard columns (Dates, Quantity, etc.)
df_full['Transaction_ID'] = df_full['Transaction_ID'].fillna('Unknown')
df_full['Customer_ID'] = df_full['Customer_ID'].fillna('Unknown')
df_full['Transaction_Date'] = pd.to_datetime(df_full['Transaction_Date'], errors='coerce').fillna(pd.Timestamp('1900-01-01'))
df_full['Quantity'] = pd.to_numeric(df_full['Quantity'], errors='coerce').fillna(0).abs()

# 4. Clean the Price column (now safe to make absolute values/fill zeros)
df_full['Price'] = raw_price_numeric.fillna(0).abs()

# 5. Standardize Text Columns
df_full['Payment_Method'] = df_full['Payment_Method'].astype(str).str.replace(' ', '').str.lower()
product_mapping = {
    'Cof': 'Coffee Machine', 'Coffe': 'Coffee Machine', 'Coffee M': 'Coffee Machine', 'Coffee Ma': 'Coffee Machine',
    'Smar': 'Smartphone', 'Smartp': 'Smartphone', 'Smartpho': 'Smartphone',
    'Hea': 'Headphones', 'Headp': 'Headphones', 'Headpho': 'Headphones',
    'Tab': 'Tablet', 'Tabl': 'Tablet', 'Table': 'Tablet', 'T': 'Tablet',
    'Lap': 'Laptop', 'Lapt': 'Laptop', 'Lapto': 'Laptop'
}
df_full['Product_Name'] = df_full['Product_Name'].replace(product_mapping)
df_full['Transaction_Status'] = df_full['Transaction_Status'].astype(str).str.strip().str.capitalize().replace({'Complete': 'Completed'})

# 6. Position 'audited_prices' directly next to 'Price'
columns = list(df_full.columns)
columns.remove('audited_prices')
price_idx = columns.index('Price')
columns.insert(price_idx + 1, 'audited_prices')
df_full = df_full[columns]

# 7. Save and Preview
df_full.to_csv("7-financial_Transactions_Final_Clean.csv", index=False)
df_full.to_csv("/content/drive/My Drive/7-financial_Transactions_Final_Clean.csv", index=False)

# Live Notebook Preview
df_full[['Transaction_ID', 'Price', 'audited_prices', 'Payment_Method']].head(10)

,Transaction_ID,Price,audited_prices,Payment_Method
0,T0001,0.000000,Missing/Invalid,paypal
1,T0002,445.342025,Negative,creditcard
2,T0003,810.993012,Valid,creditcard
3,T0004,868.608341,Valid,paypal
4,T0005,763.122449,Negative,paypal
5,T0006,0.000000,Missing/Invalid,paypal
6,Unknown,0.000000,Missing/Invalid,creditcard
7,T0008,86.921269,Negative,cash
8,T0009,461.701984,Valid,paypal
9,T0010,404.890707,Valid,creditcard


In [27]:
'Cof': 'Coffee Machine'

SyntaxError: illegal target for annotation (2529496881.py, line 1)

In [28]:
# 1. Reload raw data
df_full = pd.read_csv("7-dirty_financial_transactions.csv")

# 2. Audit raw Price column BEFORE filling NaNs
# Check for truly empty cells vs actual numbers less than 0
df_full['audited_prices'] = 'Valid'
df_full.loc[df_full['Price'].isna(), 'audited_prices'] = 'Missing/Invalid'

raw_price_numeric = pd.to_numeric(df_full['Price'], errors='coerce')
df_full.loc[raw_price_numeric < 0, 'audited_prices'] = 'Negative'
df_full.loc[raw_price_numeric.isna() & df_full['Price'].notna(), 'audited_prices'] = 'Missing/Invalid'

# 3. Clean Standard Columns
df_full['Transaction_ID'] = df_full['Transaction_ID'].fillna('Unknown')
df_full['Customer_ID'] = df_full['Customer_ID'].fillna('Unknown')

# Handle Dates and immediately flag them accurately
raw_date_coerced = pd.to_datetime(df_full['Transaction_Date'], errors='coerce')
df_full['audited_dates'] = 'Valid'
df_full.loc[raw_date_coerced.isna(), 'audited_dates'] = 'Missing/Invalid'
df_full['Transaction_Date'] = raw_date_coerced.fillna(pd.Timestamp('1900-01-01'))

# Clean Math Fields
df_full['Quantity'] = pd.to_numeric(df_full['Quantity'], errors='coerce').fillna(0).abs()
df_full['Price'] = raw_price_numeric.fillna(0).abs()

# 4. Corrected Product Mapping (Separating Coffee from Coffee Machine)
product_mapping = {
    'Cof': 'Coffee',
    'Coffe': 'Coffee',
    'Coffee M': 'Coffee Machine',
    'Coffee Ma': 'Coffee Machine',
    'Smar': 'Smartphone', 'Smartp': 'Smartphone', 'Smartpho': 'Smartphone',
    'Hea': 'Headphones', 'Headp': 'Headphones', 'Headpho': 'Headphones',
    'Tab': 'Tablet', 'Tabl': 'Tablet', 'Table': 'Tablet', 'T': 'Tablet',
    'Lap': 'Laptop', 'Lapt': 'Laptop', 'Lapto': 'Laptop'
}
df_full['Product_Name'] = df_full['Product_Name'].replace(product_mapping)

# Standardize Strings
df_full['Payment_Method'] = df_full['Payment_Method'].astype(str).str.replace(' ', '').str.lower()
df_full['Transaction_Status'] = df_full['Transaction_Status'].astype(str).str.strip().str.capitalize().replace({'Complete': 'Completed'})

# 5. Perfect Alignment Reordering
columns = list(df_full.columns)
# Position Date Audit
columns.remove('audited_dates')
columns.insert(columns.index('Transaction_Date') + 1, 'audited_dates')
# Position Price Audit
columns.remove('audited_prices')
columns.insert(columns.index('Price') + 1, 'audited_prices')

df_full = df_full[columns]

# 6. Save final production copies
df_full.to_csv("7-financial_Transactions_Final_Clean.csv", index=False)
df_full.to_csv("/content/drive/My Drive/7-financial_Transactions_Final_Clean.csv", index=False)

print(f"Verified Final Rows: {df_full.shape[0]}")
df_full.head(10)

Verified Final Rows: 100000


,Transaction_ID,Transaction_Date,audited_dates,Customer_ID,Product_Name,Quantity,Price,audited_prices,Payment_Method,Transaction_Status
0,T0001,2024-08-02,Valid,C2205,Headphones,5.0,0.000000,Missing/Invalid,paypal,Nan
1,T0002,2020-02-10,Valid,C3156,Coffee,469.0,445.342025,Negative,creditcard,Pending
2,T0003,1900-01-01,Missing/Invalid,C2919,Tablet,4.0,810.993012,Valid,creditcard,Completed
3,T0004,2020-08-17,Valid,C3009,Tablet,7.0,868.608341,Valid,paypal,Pending
4,T0005,1900-01-01,Missing/Invalid,C3488,Coffee Machine,10.0,763.122449,Negative,paypal,Completed
5,T0006,2021-10-26,Valid,C4241,Smartphone,598.0,0.000000,Missing/Invalid,paypal,Completed
6,Unknown,1900-01-01,Missing/Invalid,C1313,Laptop,10.0,0.000000,Missing/Invalid,creditcard,Completed
7,T0008,1900-01-01,Missing/Invalid,C4736,Headphones,669.0,86.921269,Negative,cash,Nan
8,T0009,1900-01-01,Missing/Invalid,C3387,Tablet,10.0,461.701984,Valid,paypal,Nan
9,T0010,1900-01-01,Missing/Invalid,C2846,Laptop,1.0,404.890707,Valid,creditcard,Pending


In [30]:
# 1. Reload raw data
df_full = pd.read_csv("7-dirty_financial_transactions.csv")

# 2. Audit raw Price column BEFORE filling NaNs
df_full['audited_prices'] = 'Valid'
df_full.loc[df_full['Price'].isna(), 'audited_prices'] = 'Missing/Invalid'

raw_price_numeric = pd.to_numeric(df_full['Price'], errors='coerce')
df_full.loc[raw_price_numeric < 0, 'audited_prices'] = 'Negative'
df_full.loc[raw_price_numeric.isna() & df_full['Price'].notna(), 'audited_prices'] = 'Missing/Invalid'

# 3. Clean Standard Columns
df_full['Transaction_ID'] = df_full['Transaction_ID'].fillna('Unknown')
df_full['Customer_ID'] = df_full['Customer_ID'].fillna('Unknown')

# Handle Dates and flag them
raw_date_coerced = pd.to_datetime(df_full['Transaction_Date'], errors='coerce')
df_full['audited_dates'] = 'Valid'
df_full.loc[raw_date_coerced.isna(), 'audited_dates'] = 'Missing/Invalid'
df_full['Transaction_Date'] = raw_date_coerced.fillna(pd.Timestamp('1900-01-01'))

# Clean Math Fields
df_full['Quantity'] = pd.to_numeric(df_full['Quantity'], errors='coerce').fillna(0).abs()

# 4. Clean Price column and STRICTLY format as dollars and cents (no dollar sign)
df_full['Price'] = raw_price_numeric.fillna(0).abs().map('{:.2f}'.format)

# 5. Corrected Product Mapping
product_mapping = {
    'Cof': 'Coffee',
    'Coffe': 'Coffee',
    'Coffee M': 'Coffee Machine',
    'Coffee Ma': 'Coffee Machine',
    'Smar': 'Smartphone', 'Smartp': 'Smartphone', 'Smartpho': 'Smartphone',
    'Hea': 'Headphones', 'Headp': 'Headphones', 'Headpho': 'Headphones',
    'Tab': 'Tablet', 'Tabl': 'Tablet', 'Table': 'Tablet', 'T': 'Tablet',
    'Lap': 'Laptop', 'Lapt': 'Laptop', 'Lapto': 'Laptop'
}
df_full['Product_Name'] = df_full['Product_Name'].replace(product_mapping)

# Standardize Strings
df_full['Payment_Method'] = df_full['Payment_Method'].astype(str).str.replace(' ', '').str.lower()
df_full['Transaction_Status'] = df_full['Transaction_Status'].astype(str).str.strip().str.capitalize().replace({'Complete': 'Completed'})

# 6. Column Alignment Reordering
columns = list(df_full.columns)
columns.remove('audited_dates')
columns.insert(columns.index('Transaction_Date') + 1, 'audited_dates')
columns.remove('audited_prices')
columns.insert(columns.index('Price') + 1, 'audited_prices')
df_full = df_full[columns]

# 7. Save final production copies
df_full.to_csv("7-financial_Transactions_Final_Clean.csv", index=False)
df_full.to_csv("/content/drive/My Drive/7-financial_Transactions_Final_Clean.csv", index=False)

print(f"Verified Final Rows: {df_full.shape[0]}")
df_full.head(10)

Verified Final Rows: 100000


,Transaction_ID,Transaction_Date,audited_dates,Customer_ID,Product_Name,Quantity,Price,audited_prices,Payment_Method,Transaction_Status
0,T0001,2024-08-02,Valid,C2205,Headphones,5.0,0.00,Missing/Invalid,paypal,Nan
1,T0002,2020-02-10,Valid,C3156,Coffee,469.0,445.34,Negative,creditcard,Pending
2,T0003,1900-01-01,Missing/Invalid,C2919,Tablet,4.0,810.99,Valid,creditcard,Completed
3,T0004,2020-08-17,Valid,C3009,Tablet,7.0,868.61,Valid,paypal,Pending
4,T0005,1900-01-01,Missing/Invalid,C3488,Coffee Machine,10.0,763.12,Negative,paypal,Completed
5,T0006,2021-10-26,Valid,C4241,Smartphone,598.0,0.00,Missing/Invalid,paypal,Completed
6,Unknown,1900-01-01,Missing/Invalid,C1313,Laptop,10.0,0.00,Missing/Invalid,creditcard,Completed
7,T0008,1900-01-01,Missing/Invalid,C4736,Headphones,669.0,86.92,Negative,cash,Nan
8,T0009,1900-01-01,Missing/Invalid,C3387,Tablet,10.0,461.70,Valid,paypal,Nan
9,T0010,1900-01-01,Missing/Invalid,C2846,Laptop,1.0,404.89,Valid,creditcard,Pending


In [31]:
df_full.head(10)

,Transaction_ID,Transaction_Date,audited_dates,Customer_ID,Product_Name,Quantity,Price,audited_prices,Payment_Method,Transaction_Status
0,T0001,2024-08-02,Valid,C2205,Headphones,5.0,0.00,Missing/Invalid,paypal,Nan
1,T0002,2020-02-10,Valid,C3156,Coffee,469.0,445.34,Negative,creditcard,Pending
2,T0003,1900-01-01,Missing/Invalid,C2919,Tablet,4.0,810.99,Valid,creditcard,Completed
3,T0004,2020-08-17,Valid,C3009,Tablet,7.0,868.61,Valid,paypal,Pending
4,T0005,1900-01-01,Missing/Invalid,C3488,Coffee Machine,10.0,763.12,Negative,paypal,Completed
5,T0006,2021-10-26,Valid,C4241,Smartphone,598.0,0.00,Missing/Invalid,paypal,Completed
6,Unknown,1900-01-01,Missing/Invalid,C1313,Laptop,10.0,0.00,Missing/Invalid,creditcard,Completed
7,T0008,1900-01-01,Missing/Invalid,C4736,Headphones,669.0,86.92,Negative,cash,Nan
8,T0009,1900-01-01,Missing/Invalid,C3387,Tablet,10.0,461.70,Valid,paypal,Nan
9,T0010,1900-01-01,Missing/Invalid,C2846,Laptop,1.0,404.89,Valid,creditcard,Pending


In [32]:
# 1. Reload raw data to catch pristine values
df_full = pd.read_csv("7-dirty_financial_transactions.csv")

# 2. Audit raw Price column BEFORE modifications
df_full['audited_prices'] = 'Valid'
df_full.loc[df_full['Price'].isna(), 'audited_prices'] = 'Missing/Invalid'

raw_price_numeric = pd.to_numeric(df_full['Price'], errors='coerce')
df_full.loc[raw_price_numeric < 0, 'audited_prices'] = 'Negative'
df_full.loc[raw_price_numeric.isna() & df_full['Price'].notna(), 'audited_prices'] = 'Missing/Invalid'

# 3. Clean Standard Columns
df_full['Transaction_ID'] = df_full['Transaction_ID'].fillna('Unknown')
df_full['Customer_ID'] = df_full['Customer_ID'].fillna('Unknown')

# Force Date formatting strictly into a string column of MM/DD/YYYY
raw_date_coerced = pd.to_datetime(df_full['Transaction_Date'], errors='coerce')
df_full['audited_dates'] = 'Valid'
df_full.loc[raw_date_coerced.isna(), 'audited_dates'] = 'Missing/Invalid'
# Use .dt.strftime to guarantee the look in the text file
df_full['Transaction_Date'] = raw_date_coerced.fillna(pd.Timestamp('1900-01-01')).dt.strftime('%m/%d/%Y')

# Clean Quantity
df_full['Quantity'] = pd.to_numeric(df_full['Quantity'], errors='coerce').fillna(0).abs()

# 4. Format Price column: Keep negatives (ex: -5.05) and leave missing as NaN string
def format_price(val):
    if pd.isna(val):
        return 'NaN'
    try:
        return f"{float(val):.2f}"
    except ValueError:
        return 'NaN'

df_full['Price'] = raw_price_numeric.map(format_price)

# 5. Product Mapping & String Standardizations
product_mapping = {
    'Cof': 'Coffee', 'Coffe': 'Coffee',
    'Coffee M': 'Coffee Machine', 'Coffee Ma': 'Coffee Machine',
    'Smar': 'Smartphone', 'Smartp': 'Smartphone', 'Smartpho': 'Smartphone',
    'Hea': 'Headphones', 'Headp': 'Headphones', 'Headpho': 'Headphones',
    'Tab': 'Tablet', 'Tabl': 'Tablet', 'Table': 'Tablet', 'T': 'Tablet',
    'Lap': 'Laptop', 'Lapt': 'Laptop', 'Lapto': 'Laptop'
}
df_full['Product_Name'] = df_full['Product_Name'].replace(product_mapping)
df_full['Payment_Method'] = df_full['Payment_Method'].astype(str).str.replace(' ', '').str.lower()
df_full['Transaction_Status'] = df_full['Transaction_Status'].astype(str).str.strip().str.capitalize().replace({'Complete': 'Completed'})

# 6. Column Reordering
columns = list(df_full.columns)
columns.remove('audited_dates')
columns.insert(columns.index('Transaction_Date') + 1, 'audited_dates')
columns.remove('audited_prices')
columns.insert(columns.index('Price') + 1, 'audited_prices')
df_full = df_full[columns]

# 7. Save outputs
df_full.to_csv("7-financial_Transactions_Final_Clean.csv", index=False)
df_full.to_csv("/content/drive/My Drive/7-financial_Transactions_Final_Clean.csv", index=False)

# Preview
df_full.head(10)

,Transaction_ID,Transaction_Date,audited_dates,Customer_ID,Product_Name,Quantity,Price,audited_prices,Payment_Method,Transaction_Status
0,T0001,08/02/2024,Valid,C2205,Headphones,5.0,NaN,Missing/Invalid,paypal,Nan
1,T0002,02/10/2020,Valid,C3156,Coffee,469.0,-445.34,Negative,creditcard,Pending
2,T0003,01/01/1900,Missing/Invalid,C2919,Tablet,4.0,810.99,Valid,creditcard,Completed
3,T0004,08/17/2020,Valid,C3009,Tablet,7.0,868.61,Valid,paypal,Pending
4,T0005,01/01/1900,Missing/Invalid,C3488,Coffee Machine,10.0,-763.12,Negative,paypal,Completed
5,T0006,10/26/2021,Valid,C4241,Smartphone,598.0,NaN,Missing/Invalid,paypal,Completed
6,Unknown,01/01/1900,Missing/Invalid,C1313,Laptop,10.0,NaN,Missing/Invalid,creditcard,Completed
7,T0008,01/01/1900,Missing/Invalid,C4736,Headphones,669.0,-86.92,Negative,cash,Nan
8,T0009,01/01/1900,Missing/Invalid,C3387,Tablet,10.0,461.70,Valid,paypal,Nan
9,T0010,01/01/1900,Missing/Invalid,C2846,Laptop,1.0,404.89,Valid,creditcard,Pending


In [33]:
import pandas as pd
import shutil
from google.colab import files

# 1. Reload the raw, untouched data
df_full = pd.read_csv("7-dirty_financial_transactions.csv")

# 2. Audit raw Price column BEFORE modifications
df_full['audited_prices'] = 'Valid'
df_full.loc[df_full['Price'].isna(), 'audited_prices'] = 'Missing/Invalid'

raw_price_numeric = pd.to_numeric(df_full['Price'], errors='coerce')
df_full.loc[raw_price_numeric < 0, 'audited_prices'] = 'Negative'
df_full.loc[raw_price_numeric.isna() & df_full['Price'].notna(), 'audited_prices'] = 'Missing/Invalid'

# 3. Clean Standard Columns
df_full['Transaction_ID'] = df_full['Transaction_ID'].fillna('Unknown')
df_full['Customer_ID'] = df_full['Customer_ID'].fillna('Unknown')

# Force Date formatting strictly into a string column of MM/DD/YYYY
raw_date_coerced = pd.to_datetime(df_full['Transaction_Date'], errors='coerce')
df_full['audited_dates'] = 'Valid'
df_full.loc[raw_date_coerced.isna(), 'audited_dates'] = 'Missing/Invalid'
df_full['Transaction_Date'] = raw_date_coerced.fillna(pd.Timestamp('1900-01-01')).dt.strftime('%m/%d/%Y')

# Clean Quantity
df_full['Quantity'] = pd.to_numeric(df_full['Quantity'], errors='coerce').fillna(0).abs()

# 4. Format Price column: Keep negatives (ex: -5.05) and leave missing as NaN string
def format_price(val):
    if pd.isna(val):
        return 'NaN'
    try:
         return f"{float(val):.2f}"
    except ValueError:
        return 'NaN'

df_full['Price'] = raw_price_numeric.map(format_price)

# 5. Product Mapping & String Standardizations
product_mapping = {
    'Cof': 'Coffee', 'Coffe': 'Coffee',
    'Coffee M': 'Coffee Machine', 'Coffee Ma': 'Coffee Machine',
    'Smar': 'Smartphone', 'Smartp': 'Smartphone', 'Smartpho': 'Smartphone',
    'Hea': 'Headphones', 'Headp': 'Headphones', 'Headpho': 'Headphones',
    'Tab': 'Tablet', 'Tabl': 'Tablet', 'Table': 'Tablet', 'T': 'Tablet',
    'Lap': 'Laptop', 'Lapt': 'Laptop', 'Lapto': 'Laptop'
}
df_full['Product_Name'] = df_full['Product_Name'].replace(product_mapping)
df_full['Payment_Method'] = df_full['Payment_Method'].astype(str).str.replace(' ', '').str.lower()
df_full['Transaction_Status'] = df_full['Transaction_Status'].astype(str).str.strip().str.capitalize().replace({'Complete': 'Completed'})

# 6. Column Reordering for Audit Alignment
columns = list(df_full.columns)
columns.remove('audited_dates')
columns.insert(columns.index('Transaction_Date') + 1, 'audited_dates')
columns.remove('audited_prices')
columns.insert(columns.index('Price') + 1, 'audited_prices')
df_full = df_full[columns]

# ==========================================
# 7. SAVE TO ALL THREE DESTINATIONS (100K ROWS)
# ==========================================

# A. Save locally to Colab Environment
final_filename = "7-financial_Transactions_Final_Clean.csv"
df_full.to_csv(final_filename, index=False)

# B. Save to Google Drive
drive_path = f"/content/drive/My Drive/{final_filename}"
df_full.to_csv(drive_path, index=False)
print(f"✔️ Successfully saved all {len(df_full)} rows to Google Drive: {drive_path}")

# C. Trigger Local Download to your PC's Downloads Folder
print("📥 Triggering browser download to your computer...")
files.download(final_filename)

# Preview live table under the cell to confirm layout changes
df_full.head(10)

✔️ Successfully saved all 100000 rows to Google Drive: /content/drive/My Drive/7-financial_Transactions_Final_Clean.csv
📥 Triggering browser download to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,Transaction_ID,Transaction_Date,audited_dates,Customer_ID,Product_Name,Quantity,Price,audited_prices,Payment_Method,Transaction_Status
0,T0001,08/02/2024,Valid,C2205,Headphones,5.0,NaN,Missing/Invalid,paypal,Nan
1,T0002,02/10/2020,Valid,C3156,Coffee,469.0,-445.34,Negative,creditcard,Pending
2,T0003,01/01/1900,Missing/Invalid,C2919,Tablet,4.0,810.99,Valid,creditcard,Completed
3,T0004,08/17/2020,Valid,C3009,Tablet,7.0,868.61,Valid,paypal,Pending
4,T0005,01/01/1900,Missing/Invalid,C3488,Coffee Machine,10.0,-763.12,Negative,paypal,Completed
5,T0006,10/26/2021,Valid,C4241,Smartphone,598.0,NaN,Missing/Invalid,paypal,Completed
6,Unknown,01/01/1900,Missing/Invalid,C1313,Laptop,10.0,NaN,Missing/Invalid,creditcard,Completed
7,T0008,01/01/1900,Missing/Invalid,C4736,Headphones,669.0,-86.92,Negative,cash,Nan
8,T0009,01/01/1900,Missing/Invalid,C3387,Tablet,10.0,461.70,Valid,paypal,Nan
9,T0010,01/01/1900,Missing/Invalid,C2846,Laptop,1.0,404.89,Valid,creditcard,Pending


In [34]:
# 1. Configure your GitHub credentials
!git config --global user.email "your-email@example.com"
!git config --global user.name "Your GitHub Username"

# 2. Clone your repository (Replace with your actual GitHub username and repository name)
!git clone https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git

# 3. Move into the repository directory
%cd YOUR_REPO_NAME

# 4. Copy the freshly cleaned 100k-row CSV from your Colab environment into the local repo folder
!cp /content/7-financial_Transactions_Final_Clean.csv .

# 5. Stage, commit, and push the final audited file to GitHub
# (Make sure to replace YOUR_GITHUB_TOKEN with your actual personal access token)
!git add 7-financial_Transactions_Final_Clean.csv
!git commit -m "Add finalized 100k-row audited financial transactions dataset for Quality Data Solutions portfolio"
!git push https://YOUR_GITHUB_TOKEN@github.com/YOUR_USERNAME/YOUR_REPO_NAME.git

Cloning into 'YOUR_REPO_NAME'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: 'YOUR_REPO_NAME'
/content
cp: '/content/7-financial_Transactions_Final_Clean.csv' and './7-financial_Transactions_Final_Clean.csv' are the same file
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


In [13]:
df.to_csv("7-financial_Transactions_FULL_100k.csv", index=False)


## 4. Save Cleaned Dataset to Google Drive

In [14]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
print(len(df))

100000


In [16]:
import shutil

# Copy the dirty file from temporary session storage directly into your Google Drive
shutil.copy("7-dirty_financial_transactions.csv", "/content/drive/My Drive/7-dirty_financial_transactions.csv")
print("Dirty dataset successfully backed up to your Google Drive!")

Dirty dataset successfully backed up to your Google Drive!


In [17]:
# Define the path in Google Drive where you want to save the file
# You can change 'My Drive/Colab Notebooks/' to your preferred folder structure.
output_path = '/content/drive/My Drive/cleaned_financial_transactions.csv'

# Save the DataFrame to a CSV file in Google Drive
# index=False prevents pandas from writing the DataFrame index as a column in the CSV
df.to_csv(output_path, index=False)

print(f"Dataset successfully saved to: {output_path}")

Dataset successfully saved to: /content/drive/My Drive/cleaned_financial_transactions.csv
